# BraTS 2020 MAD study — Run A (multi-class WT/TC/ET)

Pivot from IDRiD (too small). Reuses the **validated** 4-modality pipeline (milestone fake→real
Dice 0.72). Each generation makes 4 modalities + 3 tumor masks; a fresh U-Net scores per-class Dice
on the LOCKED test. **ET (enhancing tumor, ~17% of WT, hardest region) = the rare class** we expect
to collapse first.

**Order:** Step 1 preprocess → Step 2 run **gen-1 only (the gate)** → check ET Dice → if good, run
the recursion. Free VRAM (close other kernels) before Step 2.

## Step 1 — Preprocess (build 7-channel data; ~15-20 min, streams live)

In [ ]:
import subprocess, os
PY = r"c:\Users\Tonkid\AppData\Local\Python\pythoncore-3.12-64\python.exe"
p = subprocess.Popen([PY,"-u",r"C:\Users\Tonkid\Downloads\tstr\preprocess_brats_mad.py"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: print(line, end="", flush=True)
p.wait(); print("\n[exit", p.returncode, "]")

## Step 2 — Run the GEN-1 GATE (train gen-1 GAN + probe → per-class Dice)
Edit `KIMG`/`GENS` if you want. Default runs **gen-1 only** so you can check ET transfer before
committing to the full recursion. Launches in the background.

In [ ]:
import subprocess, os
PY = r"c:\Users\Tonkid\AppData\Local\Python\pythoncore-3.12-64\python.exe"   # self-contained; no need to run Step 1 first
# ---- knobs ----
KIMG  = 300     # GAN training length per generation (all gens identical -> no confound)
GENS  = 6       # full recursion: gen1..gen6
START = 1       # gen-1 reuses the saved kimg300 checkpoint (final.pt already placed) -> ~1h, not 5h
# ----------------
LOG = r"C:\Users\Tonkid\Downloads\brats_mad.log"
cmd = [PY,"-u",r"C:\Users\Tonkid\Downloads\tstr\brats_mad_runA.py",
       "--kimg",str(KIMG),"--gens",str(GENS),"--start",str(START)]
proc = subprocess.Popen(cmd, stdout=open(LOG,"w"), stderr=subprocess.STDOUT)
print("BraTS MAD launched, PID", proc.pid, "| kimg",KIMG,"gens",START,"..",GENS,"| log ->",LOG)
print("gen-1 SKIPS GAN training (reuses kimg300 ckpt) -> just generate+probe, ~1h to first Dice row.")
print("gens 2-6 each train a fresh GAN 300 kimg (~5h/gen). crash-safe: latest.pt every 100 kimg.")
print("WATCH: check the first metrics row (~1h) -- if gen1@300 Dice ~0.73, great; if <0.5, bump KIMG.")

## Monitor — re-run anytime

In [ ]:
import os, time, glob, matplotlib.pyplot as plt
NOISE=("MIOpen","get_global","errors generated","HIPRTC","hipoc","index_t","tid +=","^~~","cuDNN","benchmark","|")
try: raw=open(LOG).read().splitlines()
except FileNotFoundError: raw=[]
real=[l for l in raw if l.strip() and not any(k in l for k in NOISE)]
started=(time.time()-os.path.getctime(LOG))/60 if os.path.exists(LOG) else 0
changed=(time.time()-os.path.getmtime(LOG))/60 if os.path.exists(LOG) else 0
gens=[l for l in real if "GEN" in l and "DONE" in l]
ticks=[l for l in real if "kimg" in l.lower()]
print(f"running ~{started:.0f} min | log changed {changed:.1f} min ago | generations done: {len(gens)}")
for g in gens: print("  ", g)
if ticks: print("latest tick:", ticks[-1])
elif started<130: print("first-gen WARMUP (normal, up to ~1-2h)")
print("\n".join(real[-6:]))
samps=sorted(glob.glob(r"C:\Users\Tonkid\Downloads\runs\brats_mad\gen*\samples\*.png"))
if samps:
    import matplotlib.image as mpimg
    plt.figure(figsize=(9,9)); plt.imshow(mpimg.imread(samps[-1])); plt.axis("off"); plt.title("latest sample (7 channels)"); plt.show()

## Results — per-class Dice across generations (run when a gen is done)

In [ ]:
import csv, os, matplotlib.pyplot as plt
CSV=r"C:\Users\Tonkid\Downloads\results\brats_mad\metrics.csv"
if not os.path.exists(CSV):
    print("No results yet -- run the Monitor cell; come back after a generation finishes.")
    raise SystemExit
rows=list(csv.DictReader(open(CSV))); rows.sort(key=lambda r:int(r["gen"]))
g=[int(r["gen"]) for r in rows]
fig,ax=plt.subplots(1,2,figsize=(13,4.6))
for c,col in [("WT","steelblue"),("TC","seagreen"),("ET","crimson")]:
    ax[0].plot(g,[float(r[f"dice_{c}"]) for r in rows],"o-",label=c,color=col,lw=2)
    ax[1].plot(g,[float(r[f"recall_{c}"]) for r in rows],"o-",label=c,color=col,lw=2)
ax[0].set_title("Per-class Dice vs generation (ET=rare)"); ax[0].set_xlabel("generation"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title("Per-class recall vs generation"); ax[1].set_xlabel("generation"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle("BraTS MAD Run A — does ET (crimson) collapse before WT/TC?"); plt.tight_layout(); plt.show()
print("gen | WT | TC | ET  (Dice)")
for r in rows: print(f"  {r['gen']} | {r['dice_WT']} | {r['dice_TC']} | {r['dice_ET']}")
if len(rows)==1:
    print("\nGEN-1 GATE: if ET Dice is non-trivial (say >0.3), the rare class transfers -> run the recursion")
    print("(set GENS=6, START=2 in Step 2). If ET~0, ET is too hard even at gen-1 -> rethink.")

## kimg sweep — find the SHORTEST training length that still gives good Dice
Reuses gen-1's saved checkpoints (kimg 100..600) — **no GAN retraining**. For each, it generates fakes,
trains a fresh U-Net, and scores the LOCKED test. If e.g. kimg 300 ≈ kimg 600, run the recursion at
the shorter length and ~halve the 2-day run. Cost: ~40-55 min per checkpoint (~4-5 h for all six).
Launches in the background; watch `sweep.log` with the Sweep-monitor cell.

In [ ]:
import subprocess, os
PY = r"c:\Users\Tonkid\AppData\Local\Python\pythoncore-3.12-64\python.exe"
# ---- knobs ---- (defaults match the 0.733 baseline; lower for a faster, still-fair sweep)
FAKES  = 5000
EPOCHS = 30
# ----------------
SWEEP_LOG = r"C:\Users\Tonkid\Downloads\sweep.log"
cmd = [PY,"-u",r"C:\Users\Tonkid\Downloads\tstr\brats_mad_kimg_sweep.py",
       "--fakes",str(FAKES),"--epochs",str(EPOCHS)]
proc = subprocess.Popen(cmd, stdout=open(SWEEP_LOG,"w"), stderr=subprocess.STDOUT)
print("kimg sweep launched, PID", proc.pid, "| fakes",FAKES,"epochs",EPOCHS,"| log ->",SWEEP_LOG)
print("sweeps kimg 100..600; ~40-55 min each. Watch with the Sweep-monitor cell.")

In [ ]:
# Sweep-monitor + results (re-run anytime)
import os, csv, matplotlib.pyplot as plt
SWEEP_LOG = r"C:\Users\Tonkid\Downloads\sweep.log"
CSVS = r"C:\Users\Tonkid\Downloads\results\brats_mad\kimg_sweep.csv"
NOISE=("MIOpen","get_global","errors generated","HIPRTC","hipoc","index_t","tid +=","^~~","cuDNN")
try: raw=[l for l in open(SWEEP_LOG).read().splitlines() if l.strip() and not any(k in l for k in NOISE)]
except FileNotFoundError: raw=[]
print("\n".join(raw[-8:]) if raw else "no sweep log yet")
if os.path.exists(CSVS):
    rows=list(csv.DictReader(open(CSVS))); rows.sort(key=lambda r:int(r["kimg"]))
    if rows:
        k=[int(r["kimg"]) for r in rows]
        plt.figure(figsize=(8,5))
        for c,col in [("WT","steelblue"),("TC","seagreen"),("ET","crimson")]:
            plt.plot(k,[float(r[f"dice_{c}"]) for r in rows],"o-",label=c,color=col,lw=2)
        plt.plot(k,[float(r["dice_mean"]) for r in rows],"k--o",label="mean",lw=2)
        plt.xlabel("gen-1 training kimg"); plt.ylabel("fake->real Dice (locked test)")
        plt.title("How much training does a generation need?"); plt.legend(); plt.grid(alpha=.3); plt.show()
        print("kimg | mean  |  WT   |  TC   |  ET")
        for r in rows: print(f"{r['kimg']:>4} | {r['dice_mean']} | {r['dice_WT']} | {r['dice_TC']} | {r['dice_ET']}")